<a href="https://colab.research.google.com/github/KennethWu123/Kenneth-W-ML-Basics-Part-1_house-price-prediction/blob/main/Finish_GoogleCollab_ML_Basic_Kenneth_Wu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data source: Kaggle House Prices: Advanced Regression Techniques (Ames subset, ~1460 rows)
# This CSV includes SalePrice, GrLivArea, and Neighborhood used in this assignment.
# (My file: 'ames iowa housing.csv' uploaded to Colab)

In [54]:
!pip install -q pandas scikit-learn numpy

In [55]:
import pandas as pd
import os

# Use the renamed copy (recommended):
csv_path = "/content/data/ames.csv"

# If you skipped renaming, set:
# csv_path = "/content/ames iowa housing.csv"

assert os.path.isfile(csv_path), f"CSV not found at: {csv_path}"
df = pd.read_csv(csv_path)
print(df.shape)
print(df.columns.tolist()[:20])  # peek at the first 20 columns

required = {'SalePrice', 'GrLivArea', 'Neighborhood'}
missing = required - set(df.columns)
print("Missing required columns:", missing)

(1460, 81)
['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt']
Missing required columns: set()


In [56]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Keep only needed columns and rename to consistent names
work = df[['SalePrice', 'GrLivArea', 'Neighborhood']].dropna().copy()
work.rename(columns={'SalePrice':'price',
                     'GrLivArea':'square_footage',
                     'Neighborhood':'location'}, inplace=True)

# Features & target
X = work[['square_footage', 'location']]
y = work['price']

# Log-transform target for stability
log_tf = FunctionTransformer(func=np.log1p, inverse_func=np.expm1, validate=True)
y_log = log_tf.transform(y.values.reshape(-1, 1)).ravel()

# Train/test split
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Preprocess: One-Hot Encode 'location', passthrough 'square_footage'
preprocessor = ColumnTransformer(
    transformers=[
        ('loc', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['location'])
    ],
    remainder='passthrough'
)

# Pipeline: preprocessing + linear regression
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Train
model.fit(X_train, y_train_log)

# Evaluate
y_pred_log = model.predict(X_test)
rmse_log = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
r2_log = r2_score(y_test_log, y_pred_log)

# Invert to dollars for readability
y_test = np.expm1(y_test_log)
y_pred = np.expm1(y_pred_log)
rmse_dollars = np.sqrt(mean_squared_error(y_test, y_pred))
r2_dollars = r2_score(y_test, y_pred)

print("=== Test Performance ===")
print(f"RMSE (log space): {rmse_log:.4f}")
print(f"R^2  (log space): {r2_log:.4f}")
print(f"RMSE ($):         ${rmse_dollars:,.0f}")
print(f"R^2  ($):         {r2_dollars:.4f}")

=== Test Performance ===
RMSE (log space): 0.2015
R^2  (log space): 0.7824
RMSE ($):         $39,669
R^2  ($):         0.7948


In [57]:
# Get feature names back from OHE + passthrough
ohe = model.named_steps['preprocessor'].named_transformers_['loc']
ohe_features = ohe.get_feature_names_out(['location']).tolist()
feature_names = ohe_features + ['square_footage']

coefs = model.named_steps['regressor'].coef_
intercept = model.named_steps['regressor'].intercept_

print("\n=== Linear Model (log-price) Coefficients ===")
print(f"(Intercept): {intercept:.6f}  [log-dollars]")
for name, c in zip(feature_names, coefs):
    print(f"{name}: {c:.6f}  [log-dollars per unit]")

sqft_coef = coefs[-1]
print(f"\nPer +1 sq ft effect ≈ exp({sqft_coef:.6f}) - 1 = {(np.exp(sqft_coef)-1):.4%}")


=== Linear Model (log-price) Coefficients ===
(Intercept): 11.445573  [log-dollars]
location_Blmngtn: 0.161617  [log-dollars per unit]
location_Blueste: -0.187911  [log-dollars per unit]
location_BrDale: -0.362613  [log-dollars per unit]
location_BrkSide: -0.233340  [log-dollars per unit]
location_ClearCr: 0.148137  [log-dollars per unit]
location_CollgCr: 0.157790  [log-dollars per unit]
location_Crawfor: 0.073074  [log-dollars per unit]
location_Edwards: -0.244346  [log-dollars per unit]
location_Gilbert: 0.089771  [log-dollars per unit]
location_IDOTRR: -0.365217  [log-dollars per unit]
location_MeadowV: -0.381585  [log-dollars per unit]
location_Mitchel: -0.002561  [log-dollars per unit]
location_NAmes: -0.072626  [log-dollars per unit]
location_NPkVill: -0.045372  [log-dollars per unit]
location_NWAmes: 0.023627  [log-dollars per unit]
location_NoRidge: 0.266110  [log-dollars per unit]
location_NridgHt: 0.422461  [log-dollars per unit]
location_OldTown: -0.308847  [log-dollars pe

In [58]:
example_neighborhood = str(work['location'].mode().iloc[0])  # you can change this to any valid name in your data
new_house = pd.DataFrame({'square_footage':[2000], 'location':[example_neighborhood]})

pred_log = model.predict(new_house)[0]
pred_price = float(np.expm1(pred_log))
print(f"Predicted price for a 2,000 sq ft home in '{example_neighborhood}': ${pred_price:,.2f}")

Predicted price for a 2,000 sq ft home in 'NAmes': $186,915.90
